# 🧹 Text Normalization, Tokenization & Data Pipeline
**Project:** `amh-synth` — *Amharic Neural Sentiment Classification Engine*  
**Objective:** Deterministic $O(N)$ Ge'ez Orthographic Normalization, Subword BPE Tokenization, and Stratified Pipeline Processing.

---

## Overview
This notebook details the end-to-end linguistic data preparation and preprocessing pipeline:
1. **$O(N)$ Ge'ez Homophone Unification:** Standardizing 125 phonetically equivalent character variants (`ሐ/ኀ/ሀ`, `ሠ/ሰ`, `ዐ/አ`, `ፀ/ጸ`).
2. **Punctuation & Noise Reduction:** Transliterating traditional Ge'ez punctuation (`፡`, `።`, `፣`, `፤`) to ASCII and collapsing social media character elongations.
3. **Subword Tokenization Efficiency:** Measuring subword fragmentation reduction using SentencePiece / BPE tokenization.
4. **Stratified Splitting:** Preparing clean, balanced train, validation, and test datasets.

In [ ]:
# 1. Environment Setup & Dependency Imports
import os
import sys
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    import seaborn as sns
    sns.set_theme(style="darkgrid")
except ImportError:
    plt.style.use("seaborn-v0_8-darkgrid")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.preprocessor import AmharicPreprocessor
from transformers import AutoTokenizer

print("Preprocessor and tokenization environment ready.")


## 2. Orthographic Normalization Pipeline

We demonstrate step-by-step transformation of noisy Amharic text containing redundant homophones, character stretching, and Ge'ez punctuation.

In [ ]:
# 2. Step-by-Step Normalization Demonstration
raw_samples = [
    "አገልግሎቱ እጅግ በጣምምምም ጥሩ ነው፤ በጣም አመሰግናለሁ። 👍",
    "ሑሉጊዜ አይሰራም፡ ገንዘቤ ተቆርጦ አገልግሎት አላገኘሁም፡ በጣም አሳፋሪ ነው!",
    "ስብሰባው፡ነገ፡በዋናው፡አዳራሽ፡ይካሄዳል፤",
    "ዋጋው ጭስስስ ነው፡ ሰው እንዴት እንዲህ ይበዘበዛል ባክህ"
]

results = []
for text in raw_samples:
    cleaned = AmharicPreprocessor.normalize(text)
    results.append({
        "Raw Input": text,
        "Cleaned / Normalized": cleaned,
        "Raw Len": len(text),
        "Cleaned Len": len(cleaned),
        "Delta": len(text) - len(cleaned)
    })

df_demo = pd.DataFrame(results)
display(df_demo)


## 3. Subword Tokenization Efficiency & Fragmentation Analysis

We analyze token sequence lengths before and after $O(N)$ orthographic normalization using the AfriBERTa SentencePiece tokenizer.

In [ ]:
# 3. Measure Subword Token Count Reductions
model_path = os.path.join(project_root, "models", "tirsit-afriberta")
if not os.path.exists(model_path):
    model_path = "Tirsit/amharic-sentiment-afriberta"

tokenizer = AutoTokenizer.from_pretrained(model_path)

test_sentences = [
    "የደንበኞች አገልግሎታችሁ እጅግ በጣም ፈጣን እና የሚያረካ ነው፡ በጣም አመሰግናለሁ!",
    "ሲስተማችሁ ሁልጊዜ አይሰራም፡ ገንዘቤ ተቆርጦ አገልግሎት አላገኘሁም፡ በጣም አሳፋሪ ነው!",
    "ስልኩ ውበትና ምርጥ ካሜራ አለው ግን ባትሪው በፍጥነት ያልቃል።",
    "ዋጋው ጭስ ነው፡ ሰው እንዴት እንዲህ ይበዘበዛል ባክህ።",
    "ስብሰባው ነገ ከሰዓት በስምንት ሰዓት በዋናው አዳራሽ ይካሄዳል።"
]

token_stats = []
for s in test_sentences:
    raw_tokens = tokenizer.tokenize(s)
    clean_s = AmharicPreprocessor.normalize(s)
    clean_tokens = tokenizer.tokenize(clean_s)
    token_stats.append({
        "Text": s[:30] + "...",
        "Raw Token Count": len(raw_tokens),
        "Clean Token Count": len(clean_tokens),
        "Reduction (%)": f"{(1 - len(clean_tokens)/len(raw_tokens))*100:.1f}%"
    })

df_tokens = pd.DataFrame(token_stats)
display(df_tokens)


## 4. Pipeline Conclusions
1. $O(N)$ Ge'ez normalization standardizes irregular spelling variations without altering semantic valence.
2. Character elongation collapse prevents subword token fragmentation, speeding up self-attention computation and stabilizing model predictions.